# MeshVTON — Demo

Recording notebook for the project presentation video.

**Cells 1-3 are SETUP. Run them BEFORE you start recording** — they install packages,
mount Drive and load a 12B model, which takes several minutes and is not interesting to watch.

**Start the screen recording at cell 5.** Cells 5-8 are the five segments of the demo and
each one runs in seconds (except the sampler, which takes ~19 s at real speed).

Nothing here is sped up: the handbook forbids it, and the 19 s is part of the story.


In [ ]:
#@title 1) Setup — packages, repo, renderer  (PRE-RECORD)
import os, subprocess, warnings
warnings.filterwarnings("ignore")

if not os.path.exists('/content/MeshVTON'):
    !git clone -q https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git pull -q

!pip -q install "diffusers>=0.34" "peft>=0.14" lpips einops sentencepiece trimesh smplx pyrender onnxruntime
!pip -q uninstall -y pyopengl PyOpenGL-accelerate > /dev/null 2>&1
!pip -q install -q "git+https://github.com/mmatl/pyopengl.git"

import importlib.util
if importlib.util.find_spec('hmr2') is None:
    !pip -q install "git+https://github.com/shubham-goel/4D-Humans.git"
!apt-get -qq install -y libglu1-mesa libosmesa6 > /dev/null 2>&1

_probe = subprocess.run(["python", "-c",
    'import os;os.environ["PYOPENGL_PLATFORM"]="egl";'
    'import pyrender;r=pyrender.OffscreenRenderer(16,16);r.delete();print("egl-ok")'],
    capture_output=True, text=True)
os.environ['PYOPENGL_PLATFORM'] = 'egl' if 'egl-ok' in _probe.stdout else 'osmesa'

if not os.path.exists('/content/IDM-VTON'):
    !git clone -q https://github.com/yisol/IDM-VTON /content/IDM-VTON

import shutil
from huggingface_hub import hf_hub_download
for repo_path in ('humanparsing/parsing_atr.onnx',
                  'humanparsing/parsing_lip.onnx',
                  'openpose/ckpts/body_pose_model.pth'):
    local = f'/content/IDM-VTON/ckpt/{repo_path}'
    if not (os.path.exists(local) and os.path.getsize(local) > 1_000_000):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(hf_hub_download('yisol/IDM-VTON', repo_path), local)
    assert os.path.getsize(local) > 1_000_000, f'corrupt download: {local}'

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('setup ok — GL platform:', os.environ['PYOPENGL_PLATFORM'])


In [ ]:
#@title 2) Data + checkpoint  (PRE-RECORD)
import os, sys, glob, shutil
sys.path.insert(0, '/content/MeshVTON/v2')
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/MeshVTON'

!mkdir -p /content/MeshVTON/data
!unzip -q -n $D/garments_3d.zip -d /content/MeshVTON/data

!mkdir -p /content/MeshVTON/checkpoints/pretrained/smplx
!cp $D/smplx/SMPLX_NEUTRAL.* /content/MeshVTON/checkpoints/pretrained/smplx/
os.environ['SMPLX_MODEL_DIR'] = '/content/MeshVTON/checkpoints/pretrained/smplx'

from meshvton2.conditioning.body import _patch_torch_load_weights_only
import torch as _torch; _patch_torch_load_weights_only(_torch)
from hmr2.models import download_models
from hmr2.configs import CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)
smpl_dir = f"{CACHE_DIR_4DHUMANS}/data/smpl"; os.makedirs(smpl_dir, exist_ok=True)
cands = glob.glob(f'{D}/smpl/*neutral*lbs*.pkl') + glob.glob(f'{D}/smpl/SMPL_NEUTRAL.pkl')
assert cands, "SMPL neutral pkl missing -> Drive/MeshVTON/smpl/"
shutil.copy(cands[0], f"{smpl_dir}/SMPL_NEUTRAL.pkl")

# The JULY checkpoint — the regime the paper reports (textured reference, 70/30 real+synthetic).
CHECKPOINT = f"{D}/v2_outputs/stage1_july/ckpt_004000.pt"  #@param {type:"string"}
USE_TEXTURE = True   # July weights expect textured references
assert os.path.exists(CHECKPOINT), f"checkpoint not found: {CHECKPOINT}"
print('data + checkpoint ready')


In [ ]:
#@title 3) Load models  (PRE-RECORD — slow, ~2-3 min)
import sys, yaml, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, '/content/MeshVTON/v2')

from meshvton2.conditioning.body import build_hmr2_backend
from meshvton2.conditioning.builder import (
    PHOTO_GARMENT_SCALE, PHOTO_HANG_PAD, PhotoView, assert_real_impl, build_conditioning)
from meshvton2.conditioning.garment import load_garment_asset
from meshvton2.conditioning.person import PersonPreprocessor, person_square_bbox
from meshvton2.model.flux_tryon import FluxTryOnSampler

assert_real_impl()
BASE = yaml.safe_load(open('/content/MeshVTON/v2/configs/base.yaml'))
SIZE = (BASE['resolution']['height'], BASE['resolution']['width'])

# Cache in globals() so re-running never reloads the 12B model.
if 'PREP' not in globals():
    PREP = PersonPreprocessor('/content/IDM-VTON')
if 'HMR2' not in globals():
    HMR2 = build_hmr2_backend()
if 'SAMPLER' not in globals():
    SAMPLER = FluxTryOnSampler(BASE['model']['flux_fill_repo'],
                               checkpoint=CHECKPOINT,
                               prompt=BASE['model']['prompt'])

print(f'models loaded — resolution {SIZE[1]}x{SIZE[0]}')
print(f'calibrated alignment: hang_pad={PHOTO_HANG_PAD}  garment_scale={PHOTO_GARMENT_SCALE}')


---
# ▶ START THE SCREEN RECORDING HERE
---

In [ ]:
#@title Segment 1 — the two inputs  (upload your own photos)
import os, pathlib, matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

GARMENT_ID = 'upper_body/00111_Tshirt'  #@param {type:"string"}
DEMO_INDEX = 0                          #@param {type:"integer"}
REUPLOAD   = True                       #@param {type:"boolean"}

UPLOAD_DIR = pathlib.Path('/content/demo_inputs'); UPLOAD_DIR.mkdir(exist_ok=True)

if REUPLOAD or 'PERSON_IMAGES' not in globals():
    uploaded = files.upload()          # Ctrl / Cmd - click to select several files
    PERSON_IMAGES = []
    for name, data in uploaded.items():
        dst = UPLOAD_DIR / name
        dst.write_bytes(data)          # write the BYTES ourselves: files.upload() drops
        PERSON_IMAGES.append(str(dst)) # them in the cwd and renames on collision
assert PERSON_IMAGES, 'no files uploaded'

GARMENT_DIR = pathlib.Path('/content/MeshVTON/data/garments_3d') / GARMENT_ID
assert GARMENT_DIR.exists(), f'garment not found: {GARMENT_DIR}'
MESH  = sorted(GARMENT_DIR.glob('*.obj'))[0]
ASSET = load_garment_asset(MESH, garment_id=GARMENT_ID.replace('/', '__'),
                           allow_untextured=True)

# Preprocessing resizes to 768x1024 WITHOUT keeping the aspect ratio: a photo that is
# not 3:4 gets stretched, and a stretched body breaks both the pose estimate and the mask.
print('uploaded photographs')
for i, p in enumerate(PERSON_IMAGES):
    w, h = Image.open(p).size
    ar = w / h
    flag = 'ok' if abs(ar - 0.75) < 0.03 else f'WILL BE STRETCHED (crop to 3:4 first)'
    mark = ' <- used for this demo' if i == DEMO_INDEX else ''
    print(f'  [{i}] {pathlib.Path(p).name:32s} {w}x{h}  ratio {ar:.2f}  {flag}{mark}')

assert 0 <= DEMO_INDEX < len(PERSON_IMAGES), f'DEMO_INDEX out of range (0..{len(PERSON_IMAGES)-1})'
PERSON_IMAGE = PERSON_IMAGES[DEMO_INDEX]   # the one that flows through segments 2-5

n = len(PERSON_IMAGES)
fig, ax = plt.subplots(1, n + 1, figsize=(3.2 * (n + 1), 6))
ax = [ax] if n + 1 == 1 else list(ax)
for a, p in zip(ax, PERSON_IMAGES):
    a.imshow(Image.open(p))
    a.set_title('input photograph', fontsize=13)
ax[DEMO_INDEX].set_title('input photograph (selected)', fontsize=13, fontweight='bold')

tex = getattr(ASSET, 'texture', None)
if tex is not None:
    ax[-1].imshow(tex); ax[-1].set_title('3D garment mesh (texture)', fontsize=13)
else:
    ax[-1].text(.5, .5, MESH.name, ha='center', va='center', fontsize=12)
    ax[-1].set_title('3D garment mesh', fontsize=13)
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

print(f'\nmesh: {MESH.name}')
print('no product photograph, no manual alignment - one photo and one .obj')


In [ ]:
#@title Segment 2 — build the conditioning for every photo (no diffusion, seconds)
import numpy as np, matplotlib.pyplot as plt

def _show(t):                      # (3,H,W) in [-1,1] -> HxWx3 uint8
    return (((t.numpy().transpose(1, 2, 0) + 1) / 2) * 255).clip(0, 255).astype('uint8')

PPS, BUNDLES = [], []
for path in PERSON_IMAGES:
    pp = PREP.process(path, size=SIZE)
    params = HMR2(pp.image, bbox=person_square_bbox(pp))
    bundle = build_conditioning(
        pp.image, params, ASSET, PhotoView(), size=SIZE, person_prep=pp,
        hang_pad=PHOTO_HANG_PAD, garment_scale=PHOTO_GARMENT_SCALE,
        use_texture=USE_TEXTURE)
    PPS.append(pp); BUNDLES.append(bundle)

titles = ['input photograph', 'normal map', 'depth + silhouette', 'appearance reference']
n = len(PERSON_IMAGES)
fig, axes = plt.subplots(n, 4, figsize=(14, 4.6 * n), squeeze=False)
for r, (pp, b) in enumerate(zip(PPS, BUNDLES)):
    for c, img in enumerate([pp.image, _show(b.control_normal),
                             _show(b.control_depth_sil), _show(b.appearance_ref)]):
        axes[r][c].imshow(img); axes[r][c].axis('off')
        if r == 0:
            axes[r][c].set_title(titles[c], fontsize=14)
plt.tight_layout(); plt.show()

print(f'{n} photographs conditioned with the same function used in training')
print('geometry   -> normal + depth + silhouette   (no colour)')
print('appearance -> flat-lit render               (colour and pattern)')


In [ ]:
#@title Segment 3 + 4 — sample every photo (28 steps each, real time)
import time, matplotlib.pyplot as plt

n = len(BUNDLES)
print(f'{n} photographs x 28 steps — about {n * 19} s in total\n')

RESULTS, t0 = [], time.time()
for i, b in enumerate(BUNDLES):
    t = time.time()
    RESULTS.append(SAMPLER.sample(b, steps=28, seed=0, control_scale=1.0))
    print(f'  [{i + 1}/{n}] {time.time() - t:.1f} s')
total = time.time() - t0

fig, axes = plt.subplots(n, 2, figsize=(7.5, 4.6 * n), squeeze=False)
for r, (pp, res) in enumerate(zip(PPS, RESULTS)):
    axes[r][0].imshow(pp.image); axes[r][0].axis('off')
    axes[r][1].imshow(res);      axes[r][1].axis('off')
    if r == 0:
        axes[r][0].set_title('input', fontsize=15)
        axes[r][1].set_title('MeshVTON', fontsize=15)
plt.tight_layout(); plt.show()

print(f'\n{total:.1f} s for {n} images — {total / n:.1f} s each')
print('everything outside the inpainting mask is carried over from the original photograph')


In [ ]:
#@title Segment 5 — the ablation: control OFF vs ON, on every photo
import time, matplotlib.pyplot as plt

n = len(BUNDLES)
print(f'geometry branch disabled — {n} more runs, about {n * 19} s\n')

OFFS = []
for i, b in enumerate(BUNDLES):
    t = time.time()
    OFFS.append(SAMPLER.sample(b, steps=28, seed=0, control_scale=0.0))
    print(f'  [{i + 1}/{n}] {time.time() - t:.1f} s')

fig, axes = plt.subplots(n, 3, figsize=(11, 4.6 * n), squeeze=False)
for r, (pp, off, on) in enumerate(zip(PPS, OFFS, RESULTS)):
    for c, img in enumerate([pp.image, off, on]):
        axes[r][c].imshow(img); axes[r][c].axis('off')
    if r == 0:
        axes[r][0].set_title('input', fontsize=15)
        axes[r][1].set_title('control OFF  (no geometry)', fontsize=15)
        axes[r][2].set_title('control ON   (geometry)', fontsize=15)
plt.tight_layout(); plt.show()

print('\nzeroing the control input recovers exactly the no-geometry model (zero-init)')
print('this difference is what mesh specificity measures: +0.088 -> +0.229')
